## Praktikum 2
##### Erweiterte Operationen auf Gesichtern

In diesem Praktikum betrachten wir weitere Operationen auf Gesichtern und trainieren erste kleine Netzwerke zur Gesichtserkennung.

[OpenFace](http://reports-archive.adm.cs.cmu.edu/anon/2016/CMU-CS-16-118.pdf)

[OpenCVs DNN-Sample für OpenFace](https://github.com/opencv/opencv/blob/master/samples/dnn/js_face_recognition.html)

[OpenCVs FaceRec-Tutorial](https://docs.opencv.org/3.3.1/da/d60/tutorial_face_main.html)

[OpenCVs FaceRec-Dokumentation](https://docs.opencv.org/3.3.1/db/d7c/group__face.html)

<div style="padding: 10px; border-left: 5px solid #B777FA; background-color: #5C18FA; border-radius: 4px">
  🐍 Bitte nutzen Sie die virtuelle Umgebung: <b>DLACV_2526</b>
</div>

In [1]:
# imports
import os, sys, dlib
import cv2
import math
import numpy as np
import utils.faceBlendCommon as fbc
import utils.faceUtils as faceUtils
import matplotlib.pyplot as plt
import _pickle as cPickle
import matplotlib
from utils.dataPath import DATA_PATH
%matplotlib inline

In [2]:
# matplotlib-Einstellungen
matplotlib.rcParams['figure.figsize'] = (6.0,6.0)
matplotlib.rcParams['image.cmap'] = 'gray'
matplotlib.rcParams['image.interpolation'] = 'bilinear'

# Konstanten
WIN_NAME = "Gesichtserkennung"
NUM_FRAMES_FOR_FPS = 100
FACE_WIDTH = 64
FACE_HEIGHT = 64
CAP_INDEX = 0   # Kamera-Index oder Videopfad
MODEL = 'e'     # Verwendetes Modell
MODEL_PATH = 'data/models/'

# Detektoren initialisieren
faceDetector = dlib.get_frontal_face_detector() 
landmarkDetector =  dlib.shape_predictor(DATA_PATH + "models/shape_predictor_68_face_landmarks.dat")

# Labels laden
labelsMap = np.load(MODEL_PATH + 'labels_map.pkl', allow_pickle=True)

<div style="
    padding: 12px; 
    border-left: 5px solid #e8f4fc; 
    background-color: #3498; 
    border-radius: 4px;
">
  📌 <b>Aufgabe 3.4:</b><br>
  • Probieren Sie die Gesichtserkennung über eine Webcam oder ein Video aus.<br>
  • Verwenden Sie jedes Model mindestens einmal.
</div>
<div style="
    padding: 12px; 
    border-left: 5px solid #e8f4fc; 
    background-color: #66611F; 
    border-radius: 4px; 
    margin-top: 5px;
">
💡 <i><b>Hinweis:</b></i> Um das verwendete Model zu verändern, bearbeiten Sie die Konstante <code>MODEL</code>.<br>
</div>

In [3]:
# Video initialisieren
cap = cv2.VideoCapture(CAP_INDEX)

# Erstellen und Auslesen
if MODEL == 'e': # Eigen
    faceRecognizer = cv2.face.EigenFaceRecognizer_create()
    faceRecognizer.read(MODEL_PATH + 'face_model_eigen.yml')
elif MODEL == 'f': # Fisher
    faceRecognizer = cv2.face.FisherFaceRecognizer_create()
    faceRecognizer.read(MODEL_PATH + 'face_model_fisher.yml')
elif MODEL == 'l': # LBPH
    faceRecognizer = cv2.face.LBPHFaceRecognizer_create()
    faceRecognizer.read(MODEL_PATH + 'face_model_lbph.yml')

In [ ]:
fps = 10
frameCount = 0

while(True):
  if (frameCount==0):
    t = cv2.getTickCount()

  # Frame holen
  ret,im = cap.read()

  if not ret:
    continue
  
  imGray = cv2.cvtColor(im,cv2.COLOR_BGR2GRAY)

  imHeight, imWidth = imGray.shape[:2]

  # Landmarken holen
  landmarks = fbc.getLandmarks(faceDetector, landmarkDetector, imGray)
  landmarks = np.array(landmarks)
  
  if len(landmarks) >= 68:
    x1Limit = landmarks[0][0] - (landmarks[36][0] - landmarks[0][0])
    x2Limit = landmarks[16][0] + (landmarks[16][0] - landmarks[45][0])
    y1Limit = landmarks[27][1] - 3*(landmarks[30][1] - landmarks[27][1])
    y2Limit = landmarks[8][1] + (landmarks[30][1] - landmarks[29][1])

    x1 = max(x1Limit,0)
    x2 = min(x2Limit, imWidth)
    y1 = max(y1Limit, 0)
    y2 = min(y2Limit, imHeight)
    imFace = imGray[y1:y2, x1:x2]

    alignedFace = faceUtils.alignFace(imFace, landmarks)
    alignedFace = cv2.resize(alignedFace, (FACE_HEIGHT, FACE_WIDTH))
    imFaceFloat = np.float32(alignedFace)/255.0

    predictedLabel = -1
    predictedLabel, score = faceRecognizer.predict(imFaceFloat)
    center = ( int((x1 + x2) /2), int((y1 + y2)/2) )
    radius = int((y2-y1)/2.0)
    cv2.circle(im, center, radius, (0, 255, 0), 1, lineType=cv2.LINE_AA)
    cv2.putText(im, '{} {}'.format(labelsMap[predictedLabel],score), (10,100), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)
  
  frameCount = frameCount + 1

  # FPS berechnen
  if (frameCount == NUM_FRAMES_FOR_FPS):
    t = (cv2.getTickCount()-t)/cv2.getTickFrequency()
    fps = NUM_FRAMES_FOR_FPS/t
    frameCount = 0

  # Anzeigen
  cv2.putText(im, "{:.1f}-fps".format(fps), (50, im.shape[0]-50), cv2.FONT_HERSHEY_COMPLEX, 1.5, (0, 0, 255), 3,cv2.LINE_AA)
  cv2.imshow(WIN_NAME, im)
  
  key = cv2.waitKey(25) & 0xFF

  # Programm anhalten
  if key==27 or cv2.getWindowProperty(WIN_NAME, cv2.WND_PROP_VISIBLE) <1:  # ESC oder 'x'
    break

cv2.destroyAllWindows()
cap.release()